# Notebook 07: Forward pass of the final model

We will build this model with plain Python lists:

```text
5 inputs -> N ReLU hidden neurons -> 1 output
```

## What this forward pass will do

This notebook follows one input example through a tiny neural network.

1. Start with 5 input numbers.
2. Build `hidden_raw`, one raw value for each hidden neuron.
3. Apply ReLU to get `hidden_after_relu`.
4. Use the hidden values later to make one final `prediction`.

The values `hidden_raw` and `hidden_after_relu` are cache values. A cache value is an intermediate result we save because the backward pass will need it later.


For the first derivation, use a tiny hidden layer:

```text
5 inputs -> 3 ReLU hidden neurons -> 1 output
```

| Name | Shape when `hidden_size = 3` | Meaning |
|---|---:|---|
| `inputs` | 5 numbers | One input example |
| `input_to_hidden_weights` | 3 rows × 5 numbers | One row per hidden neuron |
| `hidden_biases` | 3 numbers | One bias per hidden neuron |
| `hidden_raw` | 3 numbers | Hidden values before ReLU |
| `hidden_after_relu` | 3 numbers | Hidden values after ReLU |
| `hidden_to_output_weights` | 3 numbers | One output weight per hidden neuron |
| `output_bias` | 1 number | Final output bias |
| `prediction` | 1 number | The model’s final number |

## Step 1: deterministic initialization

The model needs starting weights and biases before it can make a prediction. A **weight** is a number on a connection, and a **bias** is an extra number added to a neuron.

We use `random.Random(seed)` so the values look random but repeat every time this notebook runs. That makes the lesson easier to debug because the same seed creates the same starting model.


In [20]:
import random

input_count = 5
hidden_size = 3
seed = 7

rng = random.Random(seed)
INPUTS = [0.2, -0.4, 0.1, 0.7, -0.3]

INPUT_TO_HIDDEN_WEIGHTS: list[list[float]] = [
    [round(rng.uniform(-0.5, 0.5), 3) for _ in range(input_count)]
    for _ in range(hidden_size)
]

HIDDEN_BIASES: list[float] = [
    round(rng.uniform(-0.5, 0.5), 3) for _ in range(hidden_size)
]

HIDDEN_TO_OUTPUT_WEIGHTS: list[float] = [
    round(rng.uniform(-0.5, 0.5), 3) for _ in range(hidden_size)
]

OUTPUT_BIAS: float = round(rng.uniform(-0.5, 0.5), 3)

INPUT_TO_HIDDEN_WEIGHTS, HIDDEN_BIASES, HIDDEN_TO_OUTPUT_WEIGHTS, OUTPUT_BIAS

([[-0.176, -0.349, 0.151, -0.428, 0.036],
  [-0.134, -0.442, 0.007, -0.463, -0.066],
  [-0.43, -0.409, -0.075, 0.327, -0.376]],
 [-0.277, 0.127, 0.448],
 [0.077, -0.103, 0.476],
 -0.453)

## Step 2: compute the hidden raw values

Each hidden neuron combines all 5 inputs into one number. For one neuron, the rule is:

```text
raw value = bias + input_1 × weight_1 + input_2 × weight_2 + ... + input_5 × weight_5
```

`INPUT_TO_HIDDEN_WEIGHTS` has one row per hidden neuron. The loop below picks one row, pairs each input with one weight, multiplies each pair, adds those products, and then adds the neuron bias.


In [21]:
hidden_raw = []

for neuron_index in range(hidden_size):
    weights_for_neuron = INPUT_TO_HIDDEN_WEIGHTS[neuron_index]
    bias_for_neuron = HIDDEN_BIASES[neuron_index]

    total = bias_for_neuron
    for input_value, weight in zip(INPUTS, weights_for_neuron):
        total += weight * input_value

    hidden_raw.append(round(total, 3))

hidden_raw

[-0.468, -0.027, 0.86]

## Checkpoint: trace one hidden neuron by hand

For hidden neuron 0, the code is computing this value:

```text
-0.277
+ (0.2 × -0.176)
+ (-0.4 × -0.349)
+ (0.1 × 0.151)
+ (0.7 × -0.428)
+ (-0.3 × 0.036)
= -0.468 after rounding
```

This is a good place to pause and check the shape idea: 5 inputs need 5 matching weights for this one neuron.


## Step 3: apply ReLU

ReLU is a simple gate:

```text
ReLU(x) = max(0, x)
```

If a hidden raw value is negative, ReLU turns it into `0.0`. If a hidden raw value is positive, ReLU keeps it. This gives us `hidden_after_relu`, which is the hidden layer output that will feed the final prediction.


In [22]:
hidden_after_relu = []

for raw_value in hidden_raw:
    hidden_after_relu.append(max(0.0, raw_value))

hidden_after_relu

[0.0, 0.0, 0.86]

## Cache values saved so far

The forward pass has now saved two important hidden-layer values:

| Cache name | What it stores | Why it matters later |
|---|---|---|
| `hidden_raw` | The hidden neuron values before ReLU | Backpropagation uses these to know which ReLU gates were open |
| `hidden_after_relu` | The hidden neuron values after ReLU | The output neuron uses these to compute the final prediction |

The next step is to combine `hidden_after_relu`, `HIDDEN_TO_OUTPUT_WEIGHTS`, and `OUTPUT_BIAS` into one numeric `prediction`.


## Step 4: compute the final prediction

The output neuron combines the ReLU hidden values into one number.

```text
prediction = output_bias
    + hidden_after_relu_1 × output_weight_1
    + hidden_after_relu_2 × output_weight_2
    + hidden_after_relu_3 × output_weight_3
```

In [23]:
prediction = OUTPUT_BIAS

for hidden_value, output_weight in zip(hidden_after_relu, HIDDEN_TO_OUTPUT_WEIGHTS):
    prediction += hidden_value * output_weight

prediction = round(prediction, 3)

prediction

-0.044

In [24]:
forward_cache = {
    "inputs": INPUTS,
    "hidden_raw": hidden_raw,
    "hidden_after_relu": hidden_after_relu,
    "prediction": prediction,
}

forward_cache

{'inputs': [0.2, -0.4, 0.1, 0.7, -0.3],
 'hidden_raw': [-0.468, -0.027, 0.86],
 'hidden_after_relu': [0.0, 0.0, 0.86],
 'prediction': -0.044}

In [25]:
assert len(INPUTS) == input_count
assert len(INPUT_TO_HIDDEN_WEIGHTS) == hidden_size
assert all(len(weight_row) == input_count for weight_row in INPUT_TO_HIDDEN_WEIGHTS)
assert len(HIDDEN_BIASES) == hidden_size
assert len(hidden_raw) == hidden_size
assert len(hidden_after_relu) == hidden_size
assert len(HIDDEN_TO_OUTPUT_WEIGHTS) == hidden_size
assert isinstance(OUTPUT_BIAS, float)
assert isinstance(prediction, float)

print("Forward-pass shape checks passed.")

Forward-pass shape checks passed.


## Forward-pass checklist

This notebook now has:
- A clear model diagram: `5 inputs -> N ReLU hidden neurons -> 1 output`
- Deterministic initialization with `random.Random(seed)`
- One hidden raw value computed for each hidden neuron
- ReLU applied to each hidden raw value
- One final numeric prediction
- A `forward_cache` with values needed later for backpropagation
- Shape checks that confirm the list sizes match the model
- Only Python lists, floats, and the standard library

## Step 5: start the backward pass from MSE

The backward pass starts with one number: how the loss changes when the prediction changes.

For this first hand-checkable example, we choose a tiny target value and use squared error:

```text
error = prediction - target
loss = error²
```

The derivative of `error²` is `2 × error`, so `d_loss_d_error` is the first local gradient. Because `error = prediction - target`, changing the prediction by a tiny amount changes the error by the same tiny amount. That gives `d_error_d_prediction = 1.0`, and the chain rule gives `d_loss_d_prediction`.

This `d_loss_d_prediction` value is the upstream signal that the rest of the backward pass will reuse.


In [26]:
target = 0.5

error = prediction - target
loss = error**2

d_loss_d_error = 2 * error
d_error_d_prediction = 1.0
d_loss_d_prediction = d_loss_d_error * d_error_d_prediction

print(f"prediction: {prediction:.3f}")
print(f"target: {target:.3f}")
print(f"error: {error:.3f}")
print(f"loss: {loss:.3f}")
print(f"d_loss_d_prediction: {d_loss_d_prediction:.3f}")

prediction: -0.044
target: 0.500
error: -0.544
loss: 0.296
d_loss_d_prediction: -1.088


## Step 6: compute output-layer parameter gradients

Now send `d_loss_d_prediction` into the final linear layer. The output layer computed the prediction by adding the output bias and one product per hidden neuron:

```text
prediction = output_bias + hidden_value_0 × output_weight_0 + ...
```

For one output weight, the local derivative is the hidden value that weight multiplied. The chain rule therefore says:

```text
d_loss_d_output_weight = d_loss_d_prediction × hidden_value
```

The output bias is added directly to the prediction, so its local derivative is `1.0`. That is why `d_loss_d_output_bias` is the same number as `d_loss_d_prediction` for this one example.


In [27]:
d_loss_d_hidden_to_output_weights: list[float] = []

for hidden_value in hidden_after_relu:
    d_prediction_d_output_weight_i = hidden_value
    d_loss_d_output_weight = d_loss_d_prediction * d_prediction_d_output_weight_i
    d_loss_d_hidden_to_output_weights.append(d_loss_d_output_weight)

d_prediction_d_output_bias = 1.0
d_loss_d_output_bias = d_loss_d_prediction * d_prediction_d_output_bias

print("Output-layer gradients")
for hidden_index, gradient in enumerate(d_loss_d_hidden_to_output_weights):
    print(f"d_loss_d_hidden_to_output_weights[{hidden_index}]: {gradient:.3f}")
print(f"d_loss_d_output_bias: {d_loss_d_output_bias:.3f}")

Output-layer gradients
d_loss_d_hidden_to_output_weights[0]: -0.000
d_loss_d_hidden_to_output_weights[1]: -0.000
d_loss_d_hidden_to_output_weights[2]: -0.936
d_loss_d_output_bias: -1.088


## Step 7: send gradients back into the hidden ReLU outputs

Next we ask a different question: how would the loss change if a hidden neuron’s ReLU output changed? This is not a parameter update yet; it is the backward signal that each hidden neuron receives from the output layer.

For each product `hidden_value × output_weight`, the local derivative with respect to `hidden_value` is `output_weight`. So the chain rule gives:

```text
d_loss_d_hidden_after_relu = d_loss_d_prediction × output_weight
```

This step can give a hidden neuron a nonzero signal even if its output-weight gradient was zero. The next ReLU-backward step will decide whether the signal passes through to `hidden_raw` or gets blocked.


In [29]:
d_loss_d_hidden_after_relu = []

for output_weight in HIDDEN_TO_OUTPUT_WEIGHTS:
    d_prediction_d_hidden_after_relu_i = output_weight
    d_loss_d_hidden_value = d_loss_d_prediction * d_prediction_d_hidden_after_relu_i
    d_loss_d_hidden_after_relu.append(d_loss_d_hidden_value)

print("Gradients sent into hidden ReLU outputs")
for hidden_index, gradient in enumerate(d_loss_d_hidden_after_relu):
    print(f"d_loss_d_hidden_after_relu[{hidden_index}]: {gradient:.3f}")

Gradients sent into hidden ReLU outputs
d_loss_d_hidden_after_relu[0]: -0.084
d_loss_d_hidden_after_relu[1]: 0.112
d_loss_d_hidden_after_relu[2]: -0.518


## Step 8: backprop through ReLU into the hidden raw values

Now we move one step earlier: from `hidden_after_relu` back to `hidden_raw`. ReLU is a gate that used the forward value of `hidden_raw` to decide what reached the next layer.

```text
hidden_after_relu = max(0.0, hidden_raw)
```

The local derivative for ReLU is simple:

| Forward `hidden_raw` value | ReLU gate | Local derivative | Backward effect |
|---:|---|---:|---|
| `> 0.0` | open | `1.0` | pass the gradient through |
| `<= 0.0` | closed | `0.0` | block the gradient |

So the chain rule is:

```text
d_loss_d_hidden_raw = d_loss_d_hidden_after_relu × d_hidden_after_relu_d_hidden_raw
```

This is still a bridge gradient, not a parameter update. It tells us which hidden neurons are allowed to send the learning signal farther back to their input weights and bias.


In [30]:
d_loss_d_hidden_raw = []

for raw_value, hidden_gradient in zip(hidden_raw, d_loss_d_hidden_after_relu):
    if raw_value > 0.0:
        d_hidden_after_relu_d_hidden_raw = 1.0
    else:
        d_hidden_after_relu_d_hidden_raw = 0.0

    d_loss_d_raw_value = hidden_gradient * d_hidden_after_relu_d_hidden_raw
    d_loss_d_hidden_raw.append(d_loss_d_raw_value)

print("Gradients after backprop through ReLU")
for hidden_index, gradient in enumerate(d_loss_d_hidden_raw):
    print(f"d_loss_d_hidden_raw[{hidden_index}]: {gradient:.3f}")

Gradients after backprop through ReLU
d_loss_d_hidden_raw[0]: -0.000
d_loss_d_hidden_raw[1]: 0.000
d_loss_d_hidden_raw[2]: -0.518


## Step 9: compute the hidden-layer parameter gradients

Now the backward signal has reached `hidden_raw`, so we can finally compute gradients for the first layer’s actual parameters. Each hidden raw value was computed from one hidden bias plus five input-weight products:

```text
hidden_raw_i = hidden_bias_i
             + input_0 × input_to_hidden_weight_i_0
             + input_1 × input_to_hidden_weight_i_1
             + ...
```

For one hidden input weight, the local derivative is the input value that weight multiplied:

```text
d_loss_d_input_to_hidden_weight = d_loss_d_hidden_raw × input_value
```

For one hidden bias, the local derivative is `1.0`, because the bias is added directly to `hidden_raw`:

```text
d_loss_d_hidden_bias = d_loss_d_hidden_raw × 1.0
```

The many zero gradients are expected in this example. Hidden neurons 0 and 1 were closed by ReLU, so their `d_loss_d_hidden_raw` values are zero; therefore all of their input-weight and bias gradients are zero for this one example. Hidden neuron 2 was open, so it is the only hidden neuron with nonzero first-layer gradients.


In [32]:
d_loss_d_input_to_hidden_weights: list[list[float]] = []
d_loss_d_hidden_biases: list[float] = []

for d_loss_d_raw_value in d_loss_d_hidden_raw:
    gradients_for_hidden_neuron = []

    for input_value in INPUTS:
        d_hidden_raw_d_input_weight = input_value
        d_loss_d_input_weight = d_loss_d_raw_value * d_hidden_raw_d_input_weight
        gradients_for_hidden_neuron.append(d_loss_d_input_weight)

    d_hidden_raw_d_hidden_bias = 1.0
    d_loss_d_hidden_bias = d_loss_d_raw_value * d_hidden_raw_d_hidden_bias

    d_loss_d_input_to_hidden_weights.append(gradients_for_hidden_neuron)
    d_loss_d_hidden_biases.append(d_loss_d_hidden_bias)

print("Input-to-hidden weight gradients")
for hidden_index, gradients_for_hidden_neuron in enumerate(
    d_loss_d_input_to_hidden_weights
):
    print(f"hidden neuron {hidden_index}:")
    for input_index, gradient in enumerate(gradients_for_hidden_neuron):
        print(
            f"  d_loss_d_input_to_hidden_weights[{hidden_index}][{input_index}]: {gradient:.3f}"
        )

print("Hidden bias gradients")
for hidden_index, gradient in enumerate(d_loss_d_hidden_biases):
    print(f"d_loss_d_hidden_biases[{hidden_index}]: {gradient:.3f}")

Input-to-hidden weight gradients
hidden neuron 0:
  d_loss_d_input_to_hidden_weights[0][0]: -0.000
  d_loss_d_input_to_hidden_weights[0][1]: 0.000
  d_loss_d_input_to_hidden_weights[0][2]: -0.000
  d_loss_d_input_to_hidden_weights[0][3]: -0.000
  d_loss_d_input_to_hidden_weights[0][4]: 0.000
hidden neuron 1:
  d_loss_d_input_to_hidden_weights[1][0]: 0.000
  d_loss_d_input_to_hidden_weights[1][1]: -0.000
  d_loss_d_input_to_hidden_weights[1][2]: 0.000
  d_loss_d_input_to_hidden_weights[1][3]: 0.000
  d_loss_d_input_to_hidden_weights[1][4]: -0.000
hidden neuron 2:
  d_loss_d_input_to_hidden_weights[2][0]: -0.104
  d_loss_d_input_to_hidden_weights[2][1]: 0.207
  d_loss_d_input_to_hidden_weights[2][2]: -0.052
  d_loss_d_input_to_hidden_weights[2][3]: -0.363
  d_loss_d_input_to_hidden_weights[2][4]: 0.155
Hidden bias gradients
d_loss_d_hidden_biases[0]: -0.000
d_loss_d_hidden_biases[1]: 0.000
d_loss_d_hidden_biases[2]: -0.518


## Complete gradient calculation for one example

This table summarizes the full one-example backward pass. Read each gradient name as a question: `d_a_d_b` means “how does `a` change if `b` changes a tiny bit?”

```text
loss -> prediction -> hidden_after_relu -> hidden_raw -> hidden weights and biases
```

All four parameter-gradient groups required for this one example have now been computed: `hidden_to_output_weights`, `output_bias`, `input_to_hidden_weights`, and `hidden_biases`.

| Gradient | Value here | Kind | What it means |
|---|---:|---|---|
| `d_loss_d_error` | `-1.088` | bridge | How the loss changes when the signed error changes |
| `d_error_d_prediction` | `1.000` | local rule | How the error changes when the prediction changes |
| `d_loss_d_prediction` | `-1.088` | bridge | The main loss signal sent back from the prediction |
| `d_loss_d_hidden_to_output_weights[0]` | `0.000` | parameter gradient | Output weight 0 gets no update signal because hidden value 0 was `0.0` |
| `d_loss_d_hidden_to_output_weights[1]` | `0.000` | parameter gradient | Output weight 1 gets no update signal because hidden value 1 was `0.0` |
| `d_loss_d_hidden_to_output_weights[2]` | `-0.936` | parameter gradient | Output weight 2 gets a nonzero update signal because hidden value 2 was active |
| `d_loss_d_output_bias` | `-1.088` | parameter gradient | The output bias is added directly to the prediction |
| `d_loss_d_hidden_after_relu[0]` | `-0.084` | bridge | Signal sent back into hidden neuron 0 before the ReLU gate |
| `d_loss_d_hidden_after_relu[1]` | `0.112` | bridge | Signal sent back into hidden neuron 1 before the ReLU gate |
| `d_loss_d_hidden_after_relu[2]` | `-0.518` | bridge | Signal sent back into hidden neuron 2 before the ReLU gate |
| `d_loss_d_hidden_raw[0]` | `0.000` | bridge | ReLU blocked hidden neuron 0 because its raw value was negative |
| `d_loss_d_hidden_raw[1]` | `0.000` | bridge | ReLU blocked hidden neuron 1 because its raw value was negative |
| `d_loss_d_hidden_raw[2]` | `-0.518` | bridge | ReLU passed hidden neuron 2 because its raw value was positive |
| `d_loss_d_input_to_hidden_weights[0]` | 5 zeros | parameter gradient | All five input weights for hidden neuron 0 get zero signal because ReLU blocked the neuron |
| `d_loss_d_input_to_hidden_weights[1]` | 5 zeros | parameter gradient | All five input weights for hidden neuron 1 get zero signal because ReLU blocked the neuron |
| `d_loss_d_input_to_hidden_weights[2]` | `[-0.104, 0.207, -0.052, -0.363, 0.155]` | parameter gradient | Hidden neuron 2 gets one gradient per input value |
| `d_loss_d_hidden_biases[0]` | `0.000` | parameter gradient | Hidden bias 0 gets zero signal because ReLU blocked the neuron |
| `d_loss_d_hidden_biases[1]` | `0.000` | parameter gradient | Hidden bias 1 gets zero signal because ReLU blocked the neuron |
| `d_loss_d_hidden_biases[2]` | `-0.518` | parameter gradient | Hidden bias 2 gets the same signal as `d_loss_d_hidden_raw[2]` |

A **parameter gradient** is used directly in a gradient-descent update. A **bridge gradient** is not updated directly; it carries the loss signal farther backward so earlier parameters can get their own gradients. A **local rule** is the tiny derivative for one operation, such as `1.0` for an addition or the ReLU gate’s `1.0`/`0.0` decision.

The zero gradients are logical for this one example. They show that inactive ReLU paths cannot affect the loss with a tiny parameter change. Across many training examples, different hidden neurons can become active and receive nonzero gradients.
